# SQLi detection using Machine Learning and Deep Learning Models
## Proses persiapan data
pada proses ini dilakukan pengumpulan beberapa payload sql injection yang sering digunakan oleh pentester,bug hunter dari berbagai sumber, dan data berupa paragraph/string/kalimat untuk data benign. ada pun alur proses tersebut antara lain:
1. pengumpulan payload sql injection dari berbagai sumber, kemudian disimpan dalam format text.
2. proses penggabungan file payload menjadi data sql injection **payloads** ke dalam satu file
3. proses penggabungan file string/paragrap/kalimat menjadi data **benign** ke dalam satu file
3. masuk ke proses coding menggunakan python untuk cleaning data payload & benign
4. penggabungan data payload dan benign menjadi satu file, kemudian diberikan label untuk payload diberikan label **attack**, kemudian kalimat/paragrah/string dilabeli dengan **normal**

Referensi:
- Github SQL Injection Payloads (intruders); 
  - https://github.com/tungpv98/Detect-Sql-Injection-by-Machine-Learning/tree/master/Sql-Injection/source/trainingdata; 
  - https://github.com/twseptian/payloads/tree/master/other/sqli;
  - https://github.com/danielmiessler/SecLists/tree/master/Fuzzing/SQLi;
  - https://github.com/swisskyrepo/PayloadsAllTheThings/tree/master/SQL%20Injection/Intruder
- Github SQL Injection Analyzing and Coding
  - [SQLi-Detection-using-Machine-Learning](https://github.com/twseptian/SQLi-Detection-using-Machine-Learning/blob/master/SQL%20injection%20detection%20improved%20prediction%20additional%20benign%20and%20sqli%20notebook.ipynb)

In [ ]:
# load required packages/libraries, and mounting our google drive directory
import os
import subprocess
import glob
import time
import pandas as pd
from google.colab import drive # google drive libraries to load our dataset
# load ntlk libraries
import nltk
from nltk import ngrams
from nltk.tokenize import sent_tokenize
from nltk.stem import PorterStemmer
from nltk.stem import PorterStemmer
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords 
from nltk.tokenize import word_tokenize

# nltk download libraries (punkt, stopwords,wordnet)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# function to remove stopwords from ntlk
stop_words = set(stopwords.words('english'))
def fun_remove_stop_words(posts):
  filtered=''
  for x in posts.split(' '):
    if x not in stop_words:
      filtered+=' '+x
  return filtered

# dataset preparation process
def dataset():
  print("\n[+] Mounting google drive & listing directory")
  drive.mount('/content/drive') # mounting and initial our dataset directory
  # check google drive using subprocess to run linux command
  check_drive_path = subprocess.run(['ls', '-la', '/content/drive/MyDrive/Colab_Notebooks/sqlixss/codes/dataset_sqli'],
                                    stdout=subprocess.PIPE)
  print(check_drive_path.stdout.decode('utf-8'))
  # load plain text data from our google drive
  plain_text_data = pd.read_csv('drive/MyDrive/Colab_Notebooks/sqlixss/codes/dataset_sqli/plain.txt', 
                                sep='Aw3s0meSc0t7', names=['benign'], header=None, engine='python')
  # load sqli payload data from our google drive
  sqli_payload_data = 'drive/MyDrive/Colab_Notebooks/sqlixss/codes/dataset_sqli/sqli_payloads.txt'
  
  return plain_text_data,sqli_payload_data

# function sqli cleaning
def sqli_data_cleaning(sqli_payload_data):
  # append and replace some of chars from sqli payloads
  sqli_data=[]
  f = open(sqli_payload_data, "r")
  for x in f:
    sqli_data.append(x)
  for i in range(len(sqli_data)):
    sqli_data[i]=sqli_data[i].replace('\n', '')
    sqli_data[i]=sqli_data[i].replace('%20', ' ')
    sqli_data[i]=sqli_data[i].replace('=', ' = ')
    sqli_data[i]=sqli_data[i].replace('((', ' (( ')
    sqli_data[i]=sqli_data[i].replace('))', ' )) ')
    sqli_data[i]=sqli_data[i].replace('(', ' ( ')
    sqli_data[i]=sqli_data[i].replace(')', ' ) ')
  
  return sqli_data
  
# function plain text cleaning
def plain_text_data_cleaning(plain_text_data):
  plain_text_data=plain_text_data['benign'].values # take all benign sentence data
  # convert from list to string
  plain_data=''
  for x in plain_text_data:
    plain_data+=" " + x

  plain_data=fun_remove_stop_words(plain_data)  # remove stop words
  plain_data=plain_data.split('.')              # split sentences  
  
  # seperate words inside tags
  for i in range(len(plain_data)):
    plain_data[i]=plain_data[i].replace('<', ' <')
    plain_data[i]=plain_data[i].replace('>', '> ')
    plain_data[i]=plain_data[i].replace('=', ' = ')
  
  return plain_data

# this function to create benign training data
def create_benign_training_data(plain_data): # read from plain_data
  # initial write command to a new file
  with open('drive/MyDrive/Colab_Notebooks/sqlixss/codes/dataset_sqli/benign_for_training.txt', 'w') as filehandle:
    for listitem in plain_data:
        filehandle.write('%s\n' % listitem)
  # read self created benign data
  path='drive/MyDrive/Colab_Notebooks/sqlixss/codes/dataset_sqli/benign_for_training.txt'
  benign_dataset=[]
  f = open(path, "r")
  for x in f:
      benign_dataset.append(x)
  benign_sentences=[]
  for i in benign_dataset:
      sentences=i.split('.')    
      for sentence in sentences:
          benign_sentences.append(sentence)

  return benign_sentences

# this function to create sqli training data
def create_sqli_training_data(sqli_data): # read from sqli_data
  # initial write command to a new file
  with open('drive/MyDrive/Colab_Notebooks/sqlixss/codes/dataset_sqli/sqli_for_training.txt', 'w') as filehandle:
    for listitem in sqli_data:
        filehandle.write('%s\n' % listitem)
  # read self created sqli data
  path='drive/MyDrive/Colab_Notebooks/sqlixss/codes/dataset_sqli/sqli_for_training.txt'
  sqli_dataset=[]
  f = open(path, "r")
  for x in f:
      sqli_dataset.append(x)
  sqli_sentences = sqli_dataset
  # replace numeric values by a keyword 'numeric'    
  for i in range(len(sqli_sentences)):        
        sqli_sentences[i]=sqli_sentences[i].replace('1 ', 'numeric')
        sqli_sentences[i]=sqli_sentences[i].replace(' 1', 'numeric')
        sqli_sentences[i]=sqli_sentences[i].replace("'1 ", "'numeric ")
        sqli_sentences[i]=sqli_sentences[i].replace(" 1'", " numeric'")
        sqli_sentences[i]=sqli_sentences[i].replace('1,', 'numeric,')
        sqli_sentences[i]=sqli_sentences[i].replace("1\ ", 'numeric')
        sqli_sentences[i]=sqli_sentences[i].replace("‘1", '‘numeric')
        sqli_sentences[i]=sqli_sentences[i].replace(" 2 ", " numeric ")
        sqli_sentences[i]=sqli_sentences[i].replace(' 3 ', ' numeric ')
        sqli_sentences[i]=sqli_sentences[i].replace(' 3--', ' numeric--')
        sqli_sentences[i]=sqli_sentences[i].replace(" 4 ", ' numeric ')
        sqli_sentences[i]=sqli_sentences[i].replace(" 5 ", ' numeric ')
        sqli_sentences[i]=sqli_sentences[i].replace(' 6 ', ' numeric ')
        sqli_sentences[i]=sqli_sentences[i].replace(" 7 ", ' numeric ')
        sqli_sentences[i]=sqli_sentences[i].replace(" 8 ", ' numeric ')
        sqli_sentences[i]=sqli_sentences[i].replace('1234', ' numeric ')
        sqli_sentences[i]=sqli_sentences[i].replace("22", ' numeric ')
        sqli_sentences[i]=sqli_sentences[i].replace(" 8 ", ' numeric ')
        sqli_sentences[i]=sqli_sentences[i].replace(" 200 ", ' numeric ')
        sqli_sentences[i]=sqli_sentences[i].replace("23 ", ' numeric ')
        sqli_sentences[i]=sqli_sentences[i].replace('"1', '"numeric')
        sqli_sentences[i]=sqli_sentences[i].replace('1"', '"numeric')
        sqli_sentences[i]=sqli_sentences[i].replace("7659", 'numeric')
        sqli_sentences[i]=sqli_sentences[i].replace(" 37 ", ' numeric ')
        sqli_sentences[i]=sqli_sentences[i].replace(" 45 ", ' numeric ')    
  
  return sqli_sentences

def combine_sqli_benign(benign_sentences,sqli_sentences):
  values=[]
  # give labels to sql data
  for i in sqli_sentences:
    values.append((i,1))
  # give labels to benign data
  for i in benign_sentences:
    values.append((i,0))
  
  # convert to dataframe
  dataset_final=pd.DataFrame(values,columns=['Sentence','Label'])# Save data as csv
  # Save to final dataset
  dataset_final.to_csv('drive/MyDrive/Colab_Notebooks/sqlixss/codes/dataset_sqli/sqli_dataset_test.csv', 
                       index=False, encoding='utf-16')
  return dataset_final

def main():
  plain_text_data,sqli_payload_data = dataset()
  sqli_data = sqli_data_cleaning(sqli_payload_data)
  plain_data = plain_text_data_cleaning(plain_text_data)
  benign_sentences = create_benign_training_data(plain_data)
  sqli_sentences = create_sqli_training_data(sqli_data)
  dataset_final = combine_sqli_benign(benign_sentences,sqli_sentences)

  print(f"[+] SQL Injection payloads length: {len(sqli_data)}")
  print("[+] Looking 5 paragraph of our sql injection")
  print("\n".join(sqli_data[:5]))
  print(f"[+] Plain Text data length: {len(plain_text_data)}")
  print(f"\n[+] Plain Text data records after cleaned: {len(plain_data)}")
  print("[+] Looking 5 paragraph of our plain data after cleaned")
  print("\n".join(plain_data[:5]))
  #check length of each label on dataset
  print("\n[+] Length of Benign label on dataset: ",len(benign_sentences))
  print("[+] Length of SQLI label on dataset: ",len(sqli_sentences))
  print("\n[+] read 5 lines of our final dataset\n")
  print(dataset_final.head(5))

if __name__ == '__main__':
  main()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!

[+] Mounting google drive & listing directory
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
total 9906
-rw------- 1 root root  284032 Aug  3 10:59 benign_for_training.txt
-rw------- 1 root root  399225 Jul 15 07:41 benign.txt
-rw------- 1 root root     151 Aug  3 05:20 plain.gdoc
-rw------- 1 root root  374280 Aug  3 10:50 plain.txt
-rw------- 1 root root 4849778 Aug  3 09:53 sqli_dataset.csv
-rw------- 1 root root     151 Jul 15 08:36 sqli_dataset.gsheet
-rw------- 1 root root 2606030 Aug  3 10:59 sqli_dataset_test.csv
-rw------- 1 root root     151 Aug  3 10:45 s